# Ultimate NIDS Pipeline: Supervised Vector Space Engineering (PyTorch/CUDA)
**Goal:** Fix the "Zero Recall" on R2L/U2R by warping the feature space to maximize class separation.

**The "Unconventional" Approach: NCA + Isolation Embeddings + Autoencoding**
We upgrade from linear projections to non-linear Metric Learning, Explicit Vector Isolation, and Self-Supervised Normality Scoring.

**New Architecture:**
1.  **Manifold Mixup:** Linear Interpolation to generate high-quality synthetic R2L/U2R samples.
2.  **Neighborhood Components Analysis (NCA):** A powerful Metric Learning algorithm that learns a vector space where same-class points are spatially close.
3.  **Isolation Embeddings:** We train separate Isolation Forests for each class to provide "Membership Probability" coordinates.
4.  **Autoencoder Reconstruction (GPU):** A PyTorch neural network learns to reconstruct "Normal" traffic. The reconstruction error serves as a powerful "Weirdness Score".
5.  **Deep PyTorch Classifier (GPU):** The final classifier is a deep neural network trained on CUDA with batch normalization and dropout.

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc

# The New Stack
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Processing Unit: {device}")

# Config
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
DATA_DIR = 'Data'
CSV_FILE = os.path.join(DATA_DIR, 'network_connections.csv')
MAP_FILE = os.path.join(DATA_DIR, 'attack2category_map.txt')

# 1. Load Data & Mapping
attack_map = {'normal': 'normal'}
try:
    with open(MAP_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2: attack_map[parts[0]] = parts[1]
except FileNotFoundError: pass

df = pd.read_csv(CSV_FILE)
df['label'] = df['label'].astype(str).str.replace('.', '', regex=False)
df['category'] = df['label'].map(attack_map).fillna('other')
df.drop_duplicates(inplace=True)

# 2. Split FIRST (Prevents Data Leakage)
X = df.drop(['label', 'category'], axis=1)
y = df['category']

# Wir splitten hier schon, damit das Feature Engineering sauber getrennt ist
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Data Split. Train: {X_train_raw.shape}, Test: {X_test_raw.shape}")

del df, X, y
gc.collect()

# 3. Advanced Feature Engineering Pipeline
def engineer_features(df_in, fit=False, encoders=None):
    df = df_in.copy()
    
    # A. Interaction Features (Domain Knowledge)
    # Verhältnis Senden/Empfangen (verhindert Division durch Null mit +1)
    df['byte_ratio'] = (df['src_bytes'] + 1) / (df['dst_bytes'] + 1)
    # Verhältnis Fehler zu Verbindungen
    df['error_ratio'] = (df['serror_rate'] + df['rerror_rate']) / (df['count'] + 1)
    
    # B. Categorical: Target Encoding (Smoothed)
    # Statt Frequency nutzen wir einfache Mappings, die wir lernen
    cat_cols = ['protocol_type', 'service', 'flag']
    new_encoders = {} if fit else None
    
    for col in cat_cols:
        if fit:
            # Wir nutzen hier simple Frequency Encoding, aber SAUBER auf Train gefittet
            mapper = df[col].value_counts(normalize=True).to_dict()
            new_encoders[col] = mapper
            df[col] = df[col].map(mapper).fillna(0).astype(np.float32)
        else:
            df[col] = df[col].map(encoders[col]).fillna(0).astype(np.float32)
            
    # C. Numerical: Quantile Transformation (The Neural Net Booster)
    # Erzwingt Gaußsche Normalverteilung für ALLE numerischen Spalten
    # Das ist viel besser als Log-Transform für Deep Learning
    num_cols = [c for c in df.columns if c not in cat_cols]
    
    if fit:
        # Output distribution 'normal' macht die Daten perfekt für Autoencoder
        qt = QuantileTransformer(output_distribution='normal', n_quantiles=2000, random_state=42)
        df[num_cols] = qt.fit_transform(df[num_cols]).astype(np.float32)
        new_encoders['qt'] = qt
        new_encoders['num_cols'] = num_cols
    else:
        qt = encoders['qt']
        num_cols = encoders['num_cols']
        df[num_cols] = qt.transform(df[num_cols]).astype(np.float32)

    return df, new_encoders

# Apply Engineering
print("--- Engineering Features (Quantile Transform + Interactions) ---")
X_train_vec, encoders = engineer_features(X_train_raw, fit=True)
X_test_vec, _ = engineer_features(X_test_raw, fit=False, encoders=encoders)

# Encode Labels
le_y = LabelEncoder()
y_train_vec = le_y.fit_transform(y_train)
y_test_vec = le_y.transform(y_test)

# Clean Raw Data
del X_train_raw, X_test_raw, y_train, y_test
gc.collect()

# 4. Manifold Mixup (Improved)
print("--- Performing Manifold Mixup ---")
# Wir nutzen Pandas Index Alignment für schnelles Mixup
train_df = X_train_vec.copy()
train_df['target'] = y_train_vec

dfs = [train_df] # Originaldaten behalten!

for cls in np.unique(y_train_vec):
    cls_df = train_df[train_df['target'] == cls]
    count = len(cls_df)
    TARGET = 4000 
    
    if count < TARGET:
        needed = TARGET - count
        # Sampling indices
        idx1 = np.random.choice(cls_df.index, needed, replace=True)
        idx2 = np.random.choice(cls_df.index, needed, replace=True)
        
        # Vektorisierte Berechnung (Viel schneller)
        part1 = cls_df.loc[idx1].drop('target', axis=1).values
        part2 = cls_df.loc[idx2].drop('target', axis=1).values
        
        # Beta Distribution für Alpha ist oft besser als Uniform für Mixup
        alpha = np.random.beta(0.2, 0.2, size=(needed, 1)).astype(np.float32)
        
        synthetic = part1 * alpha + part2 * (1 - alpha)
        
        syn_df = pd.DataFrame(synthetic, columns=X_train_vec.columns)
        syn_df['target'] = cls
        dfs.append(syn_df)

train_balanced = pd.concat(dfs).sample(frac=1, random_state=42)
X_train_bal = train_balanced.drop('target', axis=1)
y_train_bal = train_balanced['target']

del train_df, dfs
gc.collect()

print(f"Balanced Training Shape: {X_train_bal.shape}")

# 5. Embedding Space Generation
print("--- Generating Vector Space (NCA + Iso + AE) ---")

# A. Convert to Numpy for Scikit-Learn
X_train_np = X_train_bal.values.astype(np.float32)
X_test_np = X_test_vec.values.astype(np.float32)

# B. Neighborhood Components Analysis (NCA)
print("1. Fitting NCA...")
nca = NeighborhoodComponentsAnalysis(n_components=16, random_state=42) # 16 Dims
idx_sample = np.random.choice(len(X_train_np), size=min(20000, len(X_train_np)), replace=False)
nca.fit(X_train_np[idx_sample], y_train_bal.iloc[idx_sample])
X_train_nca = nca.transform(X_train_np)
X_test_nca = nca.transform(X_test_np)

# C. Isolation Forests (Optimized)
print("2. Fitting Isolation Forests...")
iso_feats_train = []
iso_feats_test = []
iso_models = {}

# Wir nutzen nur die häufigsten Klassen für IsoForests, um Rauschen zu vermeiden
for cls in np.unique(y_train_bal):
    # Nur trainieren wenn genug Daten da sind
    idx_cls = (y_train_bal == cls)
    X_cls = X_train_np[idx_cls]
    
    if len(X_cls) < 100: continue
    
    iso = IsolationForest(n_estimators=100, contamination=0.01, n_jobs=-1, random_state=42)
    iso.fit(X_cls)
    iso_models[cls] = iso
    
    iso_feats_train.append(iso.decision_function(X_train_np).reshape(-1, 1))
    iso_feats_test.append(iso.decision_function(X_test_np).reshape(-1, 1))

X_train_iso = np.hstack(iso_feats_train).astype(np.float32)
X_test_iso = np.hstack(iso_feats_test).astype(np.float32)

# D. Autoencoder (Deeper & Better)
print("3. Training Autoencoder...")
class DeepAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.Mish(),
            nn.Linear(64, 32), nn.Mish(),
            nn.Linear(32, 12) # Bottleneck
        )
        self.decoder = nn.Sequential(
            nn.Linear(12, 32), nn.Mish(),
            nn.Linear(32, 64), nn.Mish(),
            nn.Linear(64, input_dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# Train on NORMAL traffic only (Semi-Supervised Anomaly Detection)
normal_cls = le_y.transform(['normal'])[0]
X_normal = X_train_np[y_train_bal == normal_cls]
# Subsample if too large
if len(X_normal) > 20000: X_normal = X_normal[np.random.choice(len(X_normal), 20000, replace=False)]

ae_model = DeepAutoencoder(X_train_np.shape[1]).to(device)
ae_opt = optim.AdamW(ae_model.parameters(), lr=0.002)
ae_loader = DataLoader(TensorDataset(torch.tensor(X_normal).to(device)), batch_size=256, shuffle=True)

for epoch in range(15):
    ae_model.train()
    for batch in ae_loader:
        x = batch[0]
        loss = nn.MSELoss()(ae_model(x), x)
        ae_opt.zero_grad()
        loss.backward()
        ae_opt.step()

# Calculate Reconstruction Error
def get_error(model, data):
    model.eval()
    loader = DataLoader(TensorDataset(torch.tensor(data).to(device)), batch_size=2048)
    errs = []
    with torch.no_grad():
        for b in loader:
            recon = model(b[0])
            # MSE per sample
            err = torch.mean((b[0] - recon)**2, dim=1).cpu().numpy()
            errs.append(err)
    return np.concatenate(errs).reshape(-1, 1)

X_train_ae = get_error(ae_model, X_train_np)
X_test_ae = get_error(ae_model, X_test_np)

# 6. Final Concatenation
# Wir nutzen Scaling nicht mehr hier, da QuantileTransformer schon 0-1/Normal liefert
X_train_final = np.hstack([X_train_np, X_train_nca, X_train_iso, X_train_ae])
X_test_final = np.hstack([X_test_np, X_test_nca, X_test_iso, X_test_ae])

print(f"Final Feature Space: {X_train_final.shape}")

Processing Unit: cuda
Data Split. Train: (100778, 41), Test: (25195, 41)
--- Engineering Features (Quantile Transform + Interactions) ---
--- Performing Manifold Mixup ---
Balanced Training Shape: (107940, 43)
--- Generating Vector Space (NCA + Iso + AE) ---
1. Fitting NCA...
2. Fitting Isolation Forests...
3. Training Autoencoder...
Final Feature Space: (107940, 65)


## 5. Training Deep Neural Network (PyTorch/CUDA)

In [3]:
# ==========================================
# ULTIMATE VECTOR SPACE PIPELINE: ARCFACE + STUDENT DISTILLATION
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import math
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Phase 0: Setup & Restoration ---")

# 1. Daten wiederherstellen & DTYPE FIX
# WICHTIG: .float() konvertiert Double (numpy default) zu Float32 (PyTorch default)
num_classes = len(np.unique(y_train_bal))

# HIER WAR DER FEHLER: Wir erzwingen float32
X_t = torch.tensor(X_train_final).float().to(device) 
y_t = torch.tensor(y_train_bal.values).long().to(device)

train_ds = TensorDataset(X_t, y_t)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
print(f"DataLoader ready. Input Dim: {X_train_final.shape[1]}. Classes: {num_classes}")

# ==========================================
# Phase 1: The ArcFace Teacher (Geometric Separation)
# ==========================================
print("\n--- Phase 1: Building Hyper-Spherical Vector Space (ArcFace) ---")

# 1. Feature Gating
class FeatureGating(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.weights = nn.Parameter(torch.ones(input_dim))
    
    def forward(self, x):
        return x * torch.sigmoid(self.weights)

# 2. ArcFace Layer
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label=None):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        # Inference / Student Training
        if label is None:
            return cosine * self.s

        # Training with Margin
        phi = cosine - self.m
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

# 3. Teacher Model
class ArcFaceTeacher(nn.Module):
    def __init__(self, input_dim, num_classes, embed_dim=128):
        super().__init__()
        self.gate = FeatureGating(input_dim)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.LeakyReLU(), nn.Dropout(0.25),
            nn.Linear(256, 256),       nn.BatchNorm1d(256), nn.LeakyReLU(), nn.Dropout(0.25),
            nn.Linear(256, embed_dim), nn.BatchNorm1d(embed_dim) 
        )
        self.arc_head = ArcMarginProduct(embed_dim, num_classes, s=30.0, m=0.5)

    def forward(self, x, label=None):
        x_gated = self.gate(x)
        embed = self.encoder(x_gated)
        embed_norm = F.normalize(embed, p=2, dim=1) 
        logits = self.arc_head(embed, label)
        return embed_norm, logits

# Teacher Training
teacher_model = ArcFaceTeacher(X_train_final.shape[1], num_classes).to(device)
optimizer_t = optim.AdamW(teacher_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler_t = optim.lr_scheduler.CosineAnnealingLR(optimizer_t, T_max=30)
criterion_ce = nn.CrossEntropyLoss()

print("Training Teacher with ArcFace...")

for epoch in range(30):
    teacher_model.train()
    total_loss = 0
    
    for xb, yb in train_dl:
        optimizer_t.zero_grad()
        # xb ist jetzt garantiert float32 durch den Fix oben
        _, logits = teacher_model(xb, yb) 
        loss = criterion_ce(logits, yb)
        loss.backward()
        optimizer_t.step()
        total_loss += loss.item()
    
    scheduler_t.step()
    if (epoch+1) % 5 == 0:
        print(f"Teacher Epoch {epoch+1} Loss: {total_loss/len(train_dl):.4f}")

teacher_model.eval()

# ==========================================
# Phase 2: Simple Student (Manifold Learning)
# ==========================================
print("\n--- Phase 2: Distilling ArcFace Space to Student ---")

class SimpleStudent(nn.Module):
    def __init__(self, input_dim, num_classes, embed_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.LeakyReLU(),
            nn.Linear(128, embed_dim)
        )
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed = self.encoder(x)
        embed = F.normalize(embed, p=2, dim=1)
        logits = self.head(embed)
        return embed, logits

student_model = SimpleStudent(X_train_final.shape[1], num_classes).to(device)
student_opt = optim.Adam(student_model.parameters(), lr=0.002)

epochs_student = 40
loss_hist = []

for epoch in range(epochs_student):
    student_model.train()
    epoch_loss = 0
    
    for xb, yb in train_dl:
        student_opt.zero_grad()
        
        s_embed, s_logits = student_model(xb)
        
        with torch.no_grad():
            t_embed, t_logits = teacher_model(xb, label=None)
            
        loss_feat = (1 - F.cosine_similarity(s_embed, t_embed, dim=1)).mean()
        loss_cls = F.cross_entropy(s_logits, yb)
        
        # 80% Vektor Matching, 20% Labels
        loss = (0.8 * loss_feat) + (0.2 * loss_cls)
        
        loss.backward()
        student_opt.step()
        epoch_loss += loss.item()
        
    loss_hist.append(epoch_loss / len(train_dl))
    if (epoch+1) % 10 == 0:
        print(f"Student Epoch {epoch+1} Loss: {loss_hist[-1]:.4f}")

plt.plot(loss_hist)
plt.title("Student ArcFace Distillation")
plt.show()

# ==========================================
# Phase 3: Validation Data Prep (NSL-KDD)
# ==========================================
print("\n--- Loading & Transforming NSL-KDD ---")

NSL_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"
NSL_COLS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

try:
    df_nsl = pd.read_csv(NSL_URL, header=None, names=NSL_COLS)
    df_nsl.drop('difficulty_level', axis=1, inplace=True)
    df_nsl['label'] = df_nsl['label'].astype(str).str.replace('.', '', regex=False)
    df_nsl['category'] = df_nsl['label'].map(attack_map).fillna('other')

    X_nsl = df_nsl.drop(['label', 'category'], axis=1)
    y_nsl = df_nsl['category']

    # --- WICHTIG: Nutze die 'engineer_features' Funktion vom vorherigen Block ---
    # Falls 'engineer_features' nicht definiert ist, müssen wir auf 'prepare_vector_input' zurückfallen,
    # aber das wäre schlecht für den Vektorraum. Ich gehe davon aus, 'engineer_features' existiert.
    if 'engineer_features' in globals():
        print("Using Advanced Feature Engineering...")
        X_nsl_vec, _ = engineer_features(X_nsl, fit=False, encoders=encoders)
    else:
        print("WARNING: Advanced Engineering function not found. Falling back (Accuracy will drop).")
        X_nsl_vec, _ = prepare_vector_input(X_nsl, fit=False, encoders=encoders)

    # Numpy Conversion
    X_nsl_np = X_nsl_vec.values.astype(np.float32)

    # 2. NCA Project
    X_nsl_nca = nca.transform(X_nsl_np)

    # 3. Isolation Forests
    iso_feats_nsl = []
    for cls in np.unique(y_train_bal): 
        if cls in iso_models:
            iso_feats_nsl.append(iso_models[cls].decision_function(X_nsl_np).reshape(-1, 1))
    X_nsl_iso = np.hstack(iso_feats_nsl).astype(np.float32)

    # 4. Deep Autoencoder Error
    # Wir nutzen 'get_error' vom vorherigen Block oder bauen es kurz nach
    def get_error_internal(model, data):
        model.eval()
        # Dtype fix auch hier
        loader = DataLoader(TensorDataset(torch.tensor(data).float().to(device)), batch_size=2048)
        errs = []
        with torch.no_grad():
            for b in loader:
                recon = model(b[0])
                err = torch.mean((b[0] - recon)**2, dim=1).cpu().numpy()
                errs.append(err)
        return np.concatenate(errs).reshape(-1, 1)
        
    X_nsl_ae = get_error_internal(ae_model, X_nsl_np)

    # 5. Concatenate
    X_nsl_final = np.hstack([X_nsl_np, X_nsl_nca, X_nsl_iso, X_nsl_ae])
    print(f"NSL-KDD Prepared. Dimensions: {X_nsl_final.shape}")

    # ==========================================
    # Phase 4: FINAL OPTIMIZED INFERENCE
    # ==========================================
    print("\n--- Running Optimized Inference (Dynamic Thresholding) ---")
    
    def predict_optimized(model, X_data, encoder):
        model.eval()
        # DTYPE FIX AUCH HIER: .float()
        loader = DataLoader(TensorDataset(torch.tensor(X_data).float()), batch_size=1024, shuffle=False)
        preds = []
        
        # Mapping Indizes
        classes = encoder.classes_
        idx_u2r = np.where(classes == 'u2r')[0][0] if 'u2r' in classes else -1
        idx_r2l = np.where(classes == 'r2l')[0][0] if 'r2l' in classes else -1
        idx_probe = np.where(classes == 'probe')[0][0] if 'probe' in classes else -1
        
        with torch.no_grad():
            for batch in loader:
                xb = batch[0].to(device)
                _, logits = model(xb) 
                
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                batch_preds = []
                for p in probs:
                    if idx_u2r != -1 and p[idx_u2r] > 0.10: batch_preds.append(idx_u2r)
                    elif idx_r2l != -1 and p[idx_r2l] > 0.15: batch_preds.append(idx_r2l)
                    elif idx_probe != -1 and p[idx_probe] > 0.25: batch_preds.append(idx_probe)
                    else: batch_preds.append(np.argmax(p))
                preds.append(np.array(batch_preds))
        return encoder.inverse_transform(np.concatenate(preds))

    # Evaluate
    print("Evaluating Student...")
    y_pred_s = predict_optimized(student_model, X_nsl_final, le_y)
    acc_s = accuracy_score(y_nsl, y_pred_s)

    print(f"\n>>> FINAL ARCFACE RESULTS <<<")
    print(f"Student Accuracy: {acc_s:.2%}")
    print("\nStudent Classification Report:")
    print(classification_report(y_nsl, y_pred_s))
    
    labels = sorted(y_nsl.unique())
    cm = confusion_matrix(y_nsl, y_pred_s, labels=labels)
    plt.figure(figsize=(10, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)
    plt.show()

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

--- Phase 0: Setup & Restoration ---
DataLoader ready. Input Dim: 65. Classes: 5

--- Phase 1: Training ArcFace Teacher ---


RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float